In [55]:
import geopandas as gpd
import pandas as pd

In [56]:
gdf = gpd.read_file("../data/oan_gems_seleccion.geojson")

In [57]:
gdf = gdf.drop(columns=["value", "unit"])
gdf = gdf.rename(columns={
    "limite_deteccion": "LD",
    "limite_cuantificacion": "LC"
})

In [58]:
gdf.head(2)

,fecha,Decision,fuente,valor_original,LD,LC,valor_transformado,chla,grupo_nombre,geometry
0,2017-01-02,None,OAN,8.900,0.100,0.1,8.900,8.900,RDP-MONTES,POINT (401164.986 6214585.025)
1,2017-01-02,si,GEMS,None,None,NaN,None,6.800000090152,LDS,POINT (678380.002 6147956.963)


In [59]:
gdf.shape

(3902, 10)

In [60]:
mask_non_numeric = pd.to_numeric(gdf['chla'], errors='coerce').isna()
chla = pd.to_numeric(gdf[~mask_non_numeric]["chla"])

In [61]:
chla

0       8.9
1       6.8
2       3.2
3       4.4
4       5.9
       ... 
3896    1.5
3897    3.0
3898    1.5
3899    1.5
3900    3.0
Name: chla, Length: 3096, dtype: float64

In [62]:
chla.min(), chla.max()

(np.float64(0.07999999797903), np.float64(470.0))

In [63]:
gdf[mask_non_numeric][["LD", "LC", "chla"]]

,LD,LC,chla
133,None,0.1,<LC
138,None,0.1,<LC
143,None,NaN,<1.00
168,None,NaN,<1.00
169,None,NaN,<1.00
...,...,...,...
3867,None,0.1,<LC
3872,None,0.1,<LC
3875,None,0.1,<LC
3895,None,0.1,<LC


In [64]:
gdf["LC"].unique()

array([0.1, nan, 1.5, 2.2])

In [65]:
gdf["LD"].unique()

array(['0.100', None, '0.600', '0.700', '0.700000000', '0.100000000'],
      dtype=object)

In [66]:
chla[chla <= 8.0].shape

(2499,)

In [67]:
chla[chla > 8.0].shape

(597,)

In [68]:
chla[chla <= 8.0].shape[0] / chla.shape[0]

0.8071705426356589

In [69]:
chla[chla > 8].shape[0] / chla.shape[0]

0.1928294573643411

In [70]:
ultraoligotrófico = chla[chla < 2.5]
oligotrofico = chla[(chla >= 2.5) & (chla < 8.0)] 
mesotrofico = chla[(chla  >= 8.0) & (chla < 25.0)]
eutrofico = chla[(chla >= 25.0) & (chla <  75)]
hipertrofico = chla[chla >= 75]

In [71]:
U = len(ultraoligotrófico) # MUY saludable
O = len(oligotrofico) # SALUDABLE
M = len(mesotrofico)# OJO
E = len(eutrofico) # PELIGRO
H = len(hipertrofico) # MUERTE

In [72]:
U, O, M, E, H 

(928, 1570, 505, 78, 15)

### Limpieza


In [73]:
gdf = gdf[~mask_non_numeric].copy()
gdf["chla"] = pd.to_numeric(gdf["chla"])
gdf.shape

(3096, 10)

In [74]:
#gdf = gdf[gdf["chla"] <= 150].copy()
#gdf.shape

In [75]:
gdf["estado_trofico"] = "E"
gdf.loc[gdf["chla"] < 2.5, "estado_trofico"] = "U"
gdf.loc[(gdf["chla"] >= 2.5) & (gdf["chla"] < 8.0), "estado_trofico"] = "O"
gdf.loc[(gdf["chla"] >= 8.0) & (gdf["chla"] < 25.0), "estado_trofico"] = "M"
gdf[["chla", "estado_trofico"]].head(10)

,chla,estado_trofico
0,8.9,M
1,6.8,O
2,3.2,O
3,4.4,O
4,5.9,O
5,5.9,O
6,11.4,M
7,4.4,O
8,5.9,O
9,6.4,O


In [76]:
gdf.groupby("estado_trofico").size()

estado_trofico
E      93
M     505
O    1570
U     928
dtype: int64

In [77]:
gdf.shape

(3096, 11)

In [78]:
gdf = gdf.drop(columns=["Decision", "valor_original", "LD", "LC", "valor_transformado"])
gdf.head(2)

,fecha,fuente,chla,grupo_nombre,geometry,estado_trofico
0,2017-01-02,OAN,8.9,RDP-MONTES,POINT (401164.986 6214585.025),M
1,2017-01-02,GEMS,6.8,LDS,POINT (678380.002 6147956.963),O


In [79]:
gdf.to_file("../data/registros_limpios_totales.geojson", driver="GeoJSON")